In [ ]:
import pandas as pd
import re
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
!pip install xgboost

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
data = pd.read_csv("IMDB Dataset.csv")
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
# initialize lemmatizer and load stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# I used a function, because I'll use the same preprocessing on every message I have
def preprocessing(text):
    if pd.isna(text):
      return ""
    # remove punctuation and marks using regex
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text) # I don't if I should remove numbers or not

    # convert to lowercase (normalization)
    text = text.lower()

    # tokenize the text using word tokenizer
    words = nltk.word_tokenize(text)

    # remove Stop words
    words = [word for word in words if word not in stop_words]

    # use lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join the words back into a single string
    return " ".join(words)

In [ ]:
data['clean_review'] = data['review'].apply(preprocessing)
data['clean_review'].head(5)

In [ ]:
data['sentiment'] = data['sentiment'].str.strip().str.lower()
data['sentiment'] = data['sentiment'].map({'negative': 0,'positive': 1})

In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(data['clean_review'])
y = data['sentiment']

print("TF-IDF Shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

y_pred = nb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss')

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))